# 🧩 Mini-Lab: LLM Observability

**Module 10: Production Deployment** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why observability is essential for debugging and monitoring LLM applications in production
2. **Implement** structured logging that captures key metadata from every LLM call (model, tokens, latency, status)
3. **Identify** what information to log for LLM-specific observability vs. traditional application logging
4. **Recognize** how structured logs enable filtering, alerting, and dashboarding

## Target Concepts

| Concept | Description |
|---------|-------------|
| Observability | Understanding the internal state and behavior of your LLM application through its external outputs — logs, metrics, and traces |
| Logging | Structured logging that captures LLM-specific metadata (tokens, latency, model, cost) for every request, enabling troubleshooting and monitoring |

## Setup

In [2]:
import json
import logging
import time
import sys
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

client = OpenAI()
MODEL = "gpt-4o-mini"
print("✅ Ready")

✅ Ready


## Why Observability for LLM Apps?

Traditional web apps are relatively predictable — the same input produces the same output. LLM applications are **non-deterministic** and have unique failure modes:

| Challenge | Why It's Hard |
|-----------|---------------|
| **Cost surprises** | A verbose prompt can cost 10x more than expected |
| **Latency spikes** | Some requests take 500ms, others 10s — why? |
| **Quality regressions** | The model "got dumber" — but when? For which queries? |
| **Silent failures** | The model returns a response, but it's wrong — no error raised |

Observability means **instrumenting your code** so you can answer questions like:
- How many tokens is each request using?
- What's the p95 latency this hour?
- Which prompts are the most expensive?
- Are error rates increasing?

The foundation of observability is **structured logging** — capturing key metadata in a machine-readable format with every LLM call.

## Step 1 — Set Up Structured Logging

Python's built-in `logging` module is the standard tool. For production LLM apps, we want **structured logs** — JSON-formatted records that are easy for log aggregation tools (e.g., Datadog, CloudWatch, ELK) to parse and query.

We'll create a logger that outputs JSON lines.

In [3]:
class JSONFormatter(logging.Formatter):
    """Format log records as JSON lines for machine-readability."""

    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            "timestamp": self.formatTime(record),
            "level": record.levelname,
            "message": record.getMessage(),
        }
        # Merge any extra fields passed via the `extra` parameter
        if hasattr(record, "extra_data"):
            log_entry.update(record.extra_data)
        return json.dumps(log_entry)


# Create a logger for our LLM service
logger = logging.getLogger("llm_service")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()  # Avoid duplicate handlers on re-run

# Output JSON logs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(JSONFormatter())
logger.addHandler(handler)

# Quick test
logger.info("Logger initialized", extra={"extra_data": {"component": "setup"}})
print("\n✅ Structured JSON logger ready")

{"timestamp": "2026-04-08 18:56:21,500", "level": "INFO", "message": "Logger initialized", "component": "setup"}

✅ Structured JSON logger ready


Notice the output is a JSON object — not a plain text string. This is what makes structured logging powerful: every field is queryable by log aggregation tools.

## Step 2 — Instrumented LLM Call

Now let's wrap our OpenAI call with logging that captures **LLM-specific metadata** on every request:

| Field | Why Log It |
|-------|------------|
| `model` | Know which model served the request |
| `prompt_tokens` | Track input cost |
| `completion_tokens` | Track output cost |
| `total_tokens` | Total usage for billing |
| `latency_ms` | Detect slow requests |
| `status` | Success vs. error |
| `error` | What went wrong (if anything) |

In [4]:
def chat_with_logging(message: str, temperature: float = 0.0) -> str:
    """
    Call the LLM and log structured metadata about the request.
    """
    start = time.perf_counter()

    log_data = {
        "model": MODEL,
        "temperature": temperature,
        "prompt_preview": message[:80],  # First 80 chars (don't log full prompts — PII risk)
    }

    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": message}],
            temperature=temperature,
        )

        latency_ms = (time.perf_counter() - start) * 1000

        log_data.update({
            "status": "success",
            "latency_ms": round(latency_ms, 1),
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "total_tokens": resp.usage.total_tokens,
        })
        logger.info("LLM call completed", extra={"extra_data": log_data})

        return resp.choices[0].message.content

    except Exception as e:
        latency_ms = (time.perf_counter() - start) * 1000
        log_data.update({
            "status": "error",
            "latency_ms": round(latency_ms, 1),
            "error_type": type(e).__name__,
            "error": str(e)[:200],
        })
        logger.error("LLM call failed", extra={"extra_data": log_data})
        raise


print("✅ chat_with_logging() defined")

✅ chat_with_logging() defined


## Step 3 — See Structured Logs in Action

Let's make a successful call and observe the log output.

In [5]:
result = chat_with_logging("What is observability in software systems? Reply in two sentences.")
print()
md(result)

{"timestamp": "2026-04-08 18:59:27,246", "level": "INFO", "message": "LLM call completed", "model": "gpt-4o-mini", "temperature": 0.0, "prompt_preview": "What is observability in software systems? Reply in two sentences.", "status": "success", "latency_ms": 1793.2, "prompt_tokens": 20, "completion_tokens": 57, "total_tokens": 77}



Observability in software systems refers to the ability to measure and understand the internal state of a system based on the data it produces, such as logs, metrics, and traces. It enables developers and operators to diagnose issues, monitor performance, and gain insights into system behavior in real-time.

Look at the JSON log line above. It contains the model, token counts, latency, and status — all in one machine-parseable record. In a real system, a log aggregator would index every field, letting you query things like:

- `status:error AND model:gpt-4o-mini` — find all errors for a specific model
- `latency_ms:>3000` — find slow requests
- `total_tokens:>1000` — find expensive requests

## Step 4 — Log Errors Too

When a call fails, the error log captures what went wrong. Let's trigger an error with a bad model name.

In [6]:
# Temporarily override MODEL to force an error
original_model = MODEL

try:
    MODEL = "non-existent-model-xyz"
    chat_with_logging("This will fail.")
except Exception:
    print("\n👆 The error was logged with structured metadata (error_type, error message, latency)")
finally:
    MODEL = original_model

{"timestamp": "2026-04-08 19:01:12,658", "level": "ERROR", "message": "LLM call failed", "model": "non-existent-model-xyz", "temperature": 0.0, "prompt_preview": "This will fail.", "status": "error", "latency_ms": 575.3, "error_type": "NotFoundError", "error": "Error code: 404 - {'error': {'message': 'The model `non-existent-model-xyz` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}"}

👆 The error was logged with structured metadata (error_type, error message, latency)


The error log includes `error_type` and `error` fields — these are invaluable for alerting ("error rate > 5% in the last 5 minutes") and post-incident debugging.

## Step 5 — Collect Metrics from Logs

In production, log aggregation tools compute metrics for you. Here, let's simulate a small workload and compute basic metrics from a simple in-memory log collector to see the kind of insights structured logs enable.

In [7]:
# Simple in-memory collector to gather metrics
request_log = []


def chat_and_collect(message: str, temperature: float = 0.0) -> str:
    """Call LLM, log it, and also collect metrics in memory."""
    start = time.perf_counter()

    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": message}],
            temperature=temperature,
        )
        latency_ms = (time.perf_counter() - start) * 1000

        request_log.append({
            "status": "success",
            "latency_ms": round(latency_ms, 1),
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "total_tokens": resp.usage.total_tokens,
        })
        return resp.choices[0].message.content

    except Exception as e:
        latency_ms = (time.perf_counter() - start) * 1000
        request_log.append({
            "status": "error",
            "latency_ms": round(latency_ms, 1),
            "error_type": type(e).__name__,
        })
        raise


print("✅ chat_and_collect() defined")

✅ chat_and_collect() defined


In [8]:
# Simulate a small workload of 4 requests
prompts = [
    "What is structured logging? One sentence.",
    "What is a metric vs. a log? One sentence.",
    "What is distributed tracing? One sentence.",
    "Explain the three pillars of observability in one sentence.",
]

for p in prompts:
    chat_and_collect(p)

print(f"✅ Made {len(request_log)} requests")

✅ Made 4 requests


In [9]:
# Compute summary metrics from collected logs
successes = [r for r in request_log if r["status"] == "success"]
errors = [r for r in request_log if r["status"] == "error"]

latencies = [r["latency_ms"] for r in successes]
total_tokens = sum(r["total_tokens"] for r in successes)
prompt_tokens = sum(r["prompt_tokens"] for r in successes)
completion_tokens = sum(r["completion_tokens"] for r in successes)

avg_latency = sum(latencies) / len(latencies) if latencies else 0
max_latency = max(latencies) if latencies else 0
min_latency = min(latencies) if latencies else 0

md(
    f"## 📊 Observability Dashboard (from {len(request_log)} requests)\n\n"
    f"| Metric | Value |\n"
    f"|--------|-------|\n"
    f"| Total requests | {len(request_log)} |\n"
    f"| Successes | {len(successes)} |\n"
    f"| Errors | {len(errors)} |\n"
    f"| Error rate | {len(errors)/len(request_log)*100:.1f}% |\n"
    f"| Avg latency | {avg_latency:.0f} ms |\n"
    f"| Min latency | {min_latency:.0f} ms |\n"
    f"| Max latency | {max_latency:.0f} ms |\n"
    f"| Total tokens | {total_tokens:,} |\n"
    f"| Prompt tokens | {prompt_tokens:,} |\n"
    f"| Completion tokens | {completion_tokens:,} |\n"
    f"| Avg tokens/request | {total_tokens/len(successes):.0f} |\n"
)

## 📊 Observability Dashboard (from 4 requests)

| Metric | Value |
|--------|-------|
| Total requests | 4 |
| Successes | 4 |
| Errors | 0 |
| Error rate | 0.0% |
| Avg latency | 965 ms |
| Min latency | 749 ms |
| Max latency | 1103 ms |
| Total tokens | 215 |
| Prompt tokens | 67 |
| Completion tokens | 148 |
| Avg tokens/request | 54 |


This is exactly the kind of dashboard that production observability tools build automatically from your structured logs. The key insight: **if you log the right fields in a structured format, the metrics come for free.**

## Step 6 — Log Levels for LLM Applications

Not all logs are equal. Python's logging module supports **levels** that let you control verbosity:

| Level | When to Use (LLM Context) | Example |
|-------|--------------------------|----------|
| `DEBUG` | Detailed troubleshooting info | Full prompt text, raw API response |
| `INFO` | Normal operations | "LLM call completed", token counts, latency |
| `WARNING` | Something unexpected but not broken | Slow response (>5s), high token usage |
| `ERROR` | Something failed | API error, timeout, invalid response |

In production, you typically set the level to `INFO` (skip `DEBUG`). During debugging, you lower it to `DEBUG` to see everything.

Let's see how to use levels meaningfully.

In [11]:
LATENCY_WARN_MS = 3000   # Warn if a request takes more than 3 seconds
TOKEN_WARN = 500         # Warn if a request uses more than 500 tokens


def chat_with_levels(message: str, temperature: float = 0.0) -> str:
    """LLM call with level-appropriate logging."""
    start = time.perf_counter()

    logger.debug("Starting LLM call", extra={"extra_data": {
        "prompt_preview": message[:80],
        "model": MODEL,
    }})

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": message}],
        temperature=temperature,
    )

    latency_ms = (time.perf_counter() - start) * 1000
    total_tokens = resp.usage.total_tokens

    log_data = {
        "model": MODEL,
        "latency_ms": round(latency_ms, 1),
        "total_tokens": total_tokens,
    }

    # Choose log level based on thresholds
    if latency_ms > LATENCY_WARN_MS:
        log_data["warning"] = "high_latency"
        logger.warning("Slow LLM response", extra={"extra_data": log_data})
    elif total_tokens > TOKEN_WARN:
        log_data["warning"] = "high_token_usage"
        logger.warning("High token usage", extra={"extra_data": log_data})
    else:
        logger.info("LLM call completed", extra={"extra_data": log_data})

    return resp.choices[0].message.content


# Normal request → INFO
print("--- Normal request (should be INFO) ---")
chat_with_levels("What is logging? One sentence.")

# Longer prompt that may use more tokens → may trigger WARNING
print("\n--- Longer request (watch the level) ---")
chat_with_levels("List and briefly explain 10 different software design patterns used in production systems.")
print()

--- Normal request (should be INFO) ---
{"timestamp": "2026-04-08 19:07:52,252", "level": "DEBUG", "message": "Starting LLM call", "prompt_preview": "What is logging? One sentence.", "model": "gpt-4o-mini"}
{"timestamp": "2026-04-08 19:07:53,205", "level": "INFO", "message": "LLM call completed", "model": "gpt-4o-mini", "latency_ms": 953.5, "total_tokens": 41}

--- Longer request (watch the level) ---
{"timestamp": "2026-04-08 19:07:53,207", "level": "DEBUG", "message": "Starting LLM call", "prompt_preview": "List and briefly explain 10 different software design patterns used in productio", "model": "gpt-4o-mini"}
{"timestamp": "2026-04-08 19:08:02,213", "level": "WARNING", "message": "Slow LLM response", "model": "gpt-4o-mini", "latency_ms": 9005.8, "total_tokens": 735, "warning": "high_latency"}



Using appropriate log levels means you can set up alerts in production like:
- **WARNING count > 10/min** → investigate latency or token spikes
- **ERROR count > 0** → page the on-call engineer

## Structured Logging vs. Print Statements

It's tempting to just use `print()` — here's why structured logging is worth the small extra effort:

| Feature | `print()` | Structured Logging |
|---------|-----------|--------------------|
| Machine-parseable | ❌ | ✅ JSON fields |
| Log levels | ❌ | ✅ DEBUG/INFO/WARN/ERROR |
| Filterable | ❌ | ✅ Query by any field |
| Timestamps | Manual | ✅ Automatic |
| Production-ready | ❌ | ✅ Integrates with log aggregators |
| Disable in prod | Difficult | ✅ Set level to WARNING |

The `print()` calls in this notebook are for *teaching clarity*. In real code, use the logger for everything.

## 🎯 Summary

### Key Takeaways

1. **LLM observability requires LLM-specific fields** — token counts, model name, latency, and status are essential metadata that traditional web app logging doesn't capture
2. **Structured logging (JSON)** — machine-parseable log records enable querying, filtering, alerting, and dashboarding in production log aggregation tools
3. **Log levels matter** — use INFO for normal operations, WARNING for anomalies (slow responses, high token usage), and ERROR for failures
4. **Metrics emerge from good logs** — if you consistently log the right fields, you can compute error rates, latency percentiles, and token usage trends without additional instrumentation